# Fire Season Timing: Validation

In [1]:
'''
05_01_REG_validation.ipynb
Validation notebook for fire season timing metrics.

Are the computed metrics correct and internally consistent?

Sections:
  Section 1: Automated sanity checks   (logical consistency of all metrics)
  Section 2: Per-ecoregion profile plots (visual validation against raw daily data)
  Section 3: Bimodality flag inspection  (flag calibration and spatial pattern)
  Section 4: Summary report              (actionable CSV for analysis decisions)

No GEE connection required. Reads entirely from disk.

Inputs  : master_<RUN_LABEL>_<RUN_VERSION>.csv
          _all_daily_counts.csv
          _eco_quality.csv
          eco_geometries.json

Outputs : plots/profile_plots/<ECO_ID>_<ECO_NAME>_climatology.png
          _validation_summary.csv
'''

import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

warnings.filterwarnings('ignore')
print('Libraries loaded.')


Libraries loaded.


In [2]:
# RUN CONFIGURATION --------------------------------------------------------------------------------
# Set these before running anything else. All output paths are derived from these values.

RUN_LABEL   = 'global'  # short name for this run
RUN_VERSION = 'v6'         # increment this for each new run
RUN_NOTES   = """
"""

In [3]:
# Folder structure and paths -----------------------------------------------------------------------

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'  # Windows
# BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac

_run_name   = f'{RUN_LABEL}_{RUN_VERSION}'
run_dir     = os.path.join(BASE_OUT_DIR, 'runs', _run_name)
raw_dir     = os.path.join(run_dir, 'raw')
output_dir  = os.path.join(run_dir, 'fire_metrics')
daily_dir   = os.path.join(output_dir, 'daily_counts')
plots_dir   = os.path.join(run_dir, 'plots')
profile_dir = os.path.join(plots_dir, 'profile_plots')

os.makedirs(plots_dir, exist_ok=True)
os.makedirs(profile_dir, exist_ok=True)

# BIMODALITY THRESHOLDS: must match values used in pipeline ----------------------
BC_THRESHOLD     = 0.555

# PLOT STYLE ----------------------------------------------------------------------
C_ONSET  = '#2ca02c'
C_PEAK   = '#d62728'
C_MEDIAN = '#ff7f0e'
C_CONC   = '#aec7e8'
C_FLAGS  = {0: '#1f77b4', 1: '#d62728'}
L_FLAGS  = {0: 'Clean (0)', 1: 'Flagged (1)'}
FIG_DPI  = 150

# Month start DOYs and labels for x-axis ticks (non-leap year)
MONTH_STARTS = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
MONTH_NAMES  = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print(f'Run       : {_run_name}')
print(f'Output dir: {output_dir}')
print(f'Plots dir: {plots_dir}')
print(f'Profiles  : {profile_dir}')

Run       : global_v6
Output dir: C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\fire_metrics
Plots dir: C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\plots
Profiles  : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\plots\profile_plots


## Load Data

In [8]:
# LOAD DATA ---------------------------------------------------------------------------------------

master_path  = os.path.join(output_dir, f'master_{_run_name}.csv')
daily_path   = os.path.join(output_dir, '_all_daily_counts.csv')
quality_path = os.path.join(output_dir, '_eco_quality.csv')
geo_path     = os.path.join(run_dir, 'eco_geometries.json')

master_df  = pd.read_csv(master_path)
daily_df   = pd.read_csv(daily_path)
quality_df = pd.read_csv(quality_path)

with open(geo_path, 'r') as f:
    geo_data = json.load(f)

# Geometry lookup: eco_id → GeoJSON geometry dict (used in map plot)
geo_lookup = {rec['eco_id']: rec['geometry'] for rec in geo_data}

# Ecoregion reference table: one stable row per ecoregion
eco_ref = master_df.groupby('eco_id').agg(
    eco_name  = ('eco_name',   'first'),
    biome_num = ('biome_num',  'first'),
    biome_name= ('biome_name', 'first'),
).reset_index()

n_eco  = master_df['eco_id'].nunique()
n_rows = len(master_df)


print('Columns in master CSV:')
for col in master_df.columns:
    print(f'  {col}')
print()
print(f'Master CSV : {n_rows} rows: {n_eco} ecoregions')
print(f'Daily CSV  : {len(daily_df)} rows')
print(f'Quality CSV: {len(quality_df)} ecoregions')
print(f'Geometries : {len(geo_data)} ecoregions')


Columns in master CSV:
  eco_id
  eco_name
  biome_num
  biome_name
  year
  onset_doy
  peak_doy
  end_doy
  season_length
  n_detections
  onset_month
  peak_month
  onset_doy_10
  end_doy_90
  onset_doy_15
  end_doy_85
  median_doy
  mean_median_div
  q25_doy
  q75_doy
  iqr_season_length
  active_days
  peak_concentration
  skewness
  kurtosis_raw
  bc
  bimodal_flag_bc
  dip_stat
  dip_pval
  bimodal_flag_dip
  bimodal_flag_year
  peak_outside_window
  n_years_valid
  pct_years_valid
  cv_peak_doy
  mean_profile_corr
  frac_flagged
  bimodal_flag_eco
  circ_mean_doy
  circ_R
  circ_rotated
  onset_doy_circ
  end_doy_circ
  season_len_circ

Master CSV : 14885 rows: 707 ecoregions
Daily CSV  : 6661993 rows
Quality CSV: 707 ecoregions
Geometries : 846 ecoregions


In [12]:
# ERA5 climatology — set era5_run_name to match the run used in 06_01
era5_run_name    = 'global_v6'   # <-- update this to your ERA5 run label
era5_clim_path   = os.path.join(BASE_OUT_DIR, 'runs', era5_run_name,
                                 'inputs', 'era5_climatology_doy.csv')

if os.path.exists(era5_clim_path):
    climatology_df = pd.read_csv(era5_clim_path)
    print(f'ERA5 climatology loaded: {len(climatology_df)} rows, '
          f'{climatology_df["eco_id"].nunique()} ecoregions')
else:
    climatology_df = None
    print(f'WARNING: ERA5 climatology not found at:\n  {era5_clim_path}')
    print('Climate sparklines will be skipped.')

ERA5 climatology loaded: 21594 rows, 59 ecoregions


## Section 1: Sanity Checks

In [5]:
# SANITY CHECKS -------------------------------------------------------------------------

checks     = []
fail_index = set()

def add_check(name, mask, description):
    n_total = len(master_df)
    n_pass  = int(mask.sum())
    n_fail  = n_total - n_pass
    rate    = n_pass / n_total * 100
    status  = 'PASS' if n_fail == 0 else 'WARN'
    checks.append({
        'Check'      : name,
        'Description': description,
        'Pass'       : n_pass,
        'Fail'       : n_fail,
        'Rate %'     : round(rate, 1),
        'Status'     : status,
    })
    if n_fail > 0:
        failing = master_df[~mask]
        for _, row in failing.iterrows():
            fail_index.add((row['eco_id'], row['year']))
        print(f'  WARN [{name}]: {n_fail} failures. First 5:')
        print(failing[['eco_id', 'eco_name', 'year']].head(5).to_string(index=False))
        print()

# --- Logical ordering of primary metrics ------------------------------------------
add_check(
    'onset_doy < end_doy',
    master_df['onset_doy'] < master_df['end_doy'],
    'Onset DOY must precede end DOY'
)
add_check(
    'season_length > 0',
    master_df['season_length'] > 0,
    'Season length must be positive'
)
add_check(
    'active_days >= 0',
    master_df['active_days'] >= 0,
    'Active days cannot be negative'
)
add_check(
    'active_days <= season_length',
    master_df['active_days'] <= master_df['season_length'],
    'Active days cannot exceed season length'
)
add_check(
    'peak_outside_window consistent',
    (
        (master_df['peak_outside_window'] == 0) &
        (master_df['peak_doy'] >= master_df['onset_doy']) &
        (master_df['peak_doy'] <= master_df['end_doy'])
    ) | (master_df['peak_outside_window'] == 1),
    'peak_outside_window flag must agree with onset/peak/end values'
)

# --- Bounded metrics --------------------------------------------------------------
add_check(
    'concentration in [0, 1]',
    (master_df['peak_concentration'] >= 0) & (master_df['peak_concentration'] <= 1),
    'Peak concentration must be between 0 and 1'
)

# --- Alternative thresholds bracket primary thresholds ----------------------------
add_check(
    'onset_10 >= onset_5',
    master_df['onset_doy_10'].fillna(master_df['onset_doy']) >= master_df['onset_doy'],
    '10% onset must be >= 5% onset'
)
add_check(
    'onset_15 >= onset_10',
    master_df['onset_doy_15'].fillna(master_df['onset_doy_10']) >=
    master_df['onset_doy_10'].fillna(master_df['onset_doy']),
    '15% onset must be >= 10% onset'
)
add_check(
    'end_90 <= end_95',
    master_df['end_doy_90'].fillna(master_df['end_doy']) <= master_df['end_doy'],
    '90% end must be <= 95% end'
)
add_check(
    'end_85 <= end_90',
    master_df['end_doy_85'].fillna(master_df['end_doy_90']) <=
    master_df['end_doy_90'].fillna(master_df['end_doy']),
    '85% end must be <= 90% end'
)

# --- IQR season length vs primary -------------------------------------------------
iqr_valid = master_df['iqr_season_length'].notna()
add_check(
    'iqr_length <= season_length',
    (~iqr_valid) | (master_df['iqr_season_length'] <= master_df['season_length']),
    'IQR season length must not exceed primary season length'
)

# --- Flag values ------------------------------------------------------------------
add_check(
    'bimodal_flag_year valid',
    master_df['bimodal_flag_year'].isin([0, 1]),
    'Per-year bimodal flag must be 0 or 1'
)
add_check(
    'bimodal_flag_eco valid',
    master_df['bimodal_flag_eco'].isin([0, 1]),
    'Ecoregion bimodal flag must be 0 or 1'
)

# --- Detection threshold ----------------------------------------------------------
add_check(
    'n_detections >= 20',
    master_df['n_detections'] >= 20,
    'All rows in master CSV should have passed MIN_DETECTIONS = 20'
)

# --- Print summary table ----------------------------------------------------------
print()
print('=' * 76)
print('SANITY CHECK SUMMARY')
print('=' * 76)
checks_df = pd.DataFrame(checks)
print(checks_df[['Check', 'Pass', 'Fail', 'Rate %', 'Status']].to_string(index=False))
print('=' * 76)
n_warn = sum(1 for c in checks if c['Status'] == 'WARN')
print(f'\n{len(checks)} checks: {len(checks) - n_warn} PASS, {n_warn} WARN')
print(f'{len(fail_index)} ecoregion-years involved in at least one failure.')

  WARN [onset_doy < end_doy]: 10 failures. First 5:
 eco_id                       eco_name  year
    369 Alaska Peninsula montane taiga  2015
    369 Alaska Peninsula montane taiga  2016
    408        Arctic foothills tundra  2009
    408        Arctic foothills tundra  2017
    413     Canadian Low Arctic tundra  2021


SANITY CHECK SUMMARY
                         Check  Pass  Fail  Rate % Status
           onset_doy < end_doy 14875    10    99.9   WARN
             season_length > 0 14885     0   100.0   PASS
              active_days >= 0 14885     0   100.0   PASS
  active_days <= season_length 14885     0   100.0   PASS
peak_outside_window consistent 14885     0   100.0   PASS
       concentration in [0, 1] 14885     0   100.0   PASS
           onset_10 >= onset_5 14885     0   100.0   PASS
          onset_15 >= onset_10 14885     0   100.0   PASS
              end_90 <= end_95 14885     0   100.0   PASS
              end_85 <= end_90 14885     0   100.0   PASS
   iqr_length <= 

## Section 2: Per-Ecoregion Profile Plots

In [ ]:
# SECTION 2: PER-ECOREGION PROFILE PLOTS ----------------------------------------------------------
# Main panel : per-year thin lines + smoothed mean profile
# Season bands: 5–95%, 10–90%, 15–85%, IQR as full-height shades + bottom bars
# Markers     : onset, peak, median, end as vertical lines
# Annotation  : stats box top-right corner

eco_ids = sorted(master_df['eco_id'].unique())

# ── COLOR SCHEME ──────────────────────────────────────────────────────────────
C_PROFILE = '#2c7bb6'
C_YR_LINE = '#aaaaaa'
C_ONSET   = '#2ca02c'
C_PEAK    = '#d62728'
C_MEDIAN  = '#ff7f0e'
C_END     = '#9467bd'
C_S95     = '#4393c3'
C_10      = '#74add1'
C_15      = '#abd9e9'
C_IQR     = '#f4a582'

for eco_id in eco_ids:
    eco_rows   = master_df[master_df['eco_id'] == eco_id]
    daily_rows = daily_df[daily_df['eco_id'] == eco_id].copy()

    eco_name   = eco_rows['eco_name'].iloc[0]
    biome_name = eco_rows['biome_name'].iloc[0]

    q_row    = quality_df[quality_df['eco_id'] == eco_id]
    flag_eco = int(q_row['bimodal_flag_eco'].iloc[0])    if len(q_row) else 0
    corr_eco = float(q_row['mean_profile_corr'].iloc[0]) if len(q_row) and pd.notna(q_row['mean_profile_corr'].iloc[0]) else np.nan
    cv_eco   = float(q_row['cv_peak_doy'].iloc[0])       if len(q_row) and pd.notna(q_row['cv_peak_doy'].iloc[0]) else np.nan

    # ── DAILY DATA (already complete — every DOY present) ─────────────────────
    years      = sorted(daily_rows['year'].unique())
    daily_full = daily_rows[['year', 'doy', 'n_detections']].copy()

    daily_full['year_total'] = daily_full.groupby('year')['n_detections'].transform('sum')
    daily_full['prop'] = np.where(
        daily_full['year_total'] > 0,
        daily_full['n_detections'] / daily_full['year_total'],
        0
    )

    # ── MEAN PROFILE (DOY 1–365 only to handle leap years) ────────────────────
    mean_prop = daily_full[daily_full['doy'] <= 365].groupby('doy')['prop'].mean().values
    smoothed  = pd.Series(mean_prop).rolling(15, center=True, min_periods=1).mean().values

    # ── VALID METRIC ROWS ─────────────────────────────────────────────────────
    m_valid = eco_rows.dropna(subset=['onset_doy', 'peak_doy', 'end_doy'])

    onset_mean  = m_valid['onset_doy'].mean()         if len(m_valid) else np.nan
    peak_mean   = m_valid['peak_doy'].mean()           if len(m_valid) else np.nan
    end_mean    = m_valid['end_doy'].mean()            if len(m_valid) else np.nan
    median_mean = m_valid['median_doy'].mean()         if 'median_doy' in m_valid.columns and len(m_valid) else np.nan
    season_len  = m_valid['season_length'].mean()      if len(m_valid) else np.nan
    iqr_len     = m_valid['iqr_season_length'].mean()  if len(m_valid) else np.nan
    active_days = m_valid['active_days'].mean()        if len(m_valid) else np.nan
    peak_conc   = m_valid['peak_concentration'].mean() if len(m_valid) else np.nan
    mm_div      = m_valid['mean_median_div'].mean()    if len(m_valid) else np.nan
    skew_val    = m_valid['skewness'].mean()           if len(m_valid) else np.nan
    bc_val      = m_valid['bc'].mean()                 if len(m_valid) else np.nan
    dip_pval    = m_valid['dip_pval'].mean()           if 'dip_pval' in m_valid.columns and len(m_valid) else np.nan
    flag_bc     = int(m_valid['bimodal_flag_bc'].mode()[0])  if 'bimodal_flag_bc' in m_valid.columns and len(m_valid) else 0
    flag_dip    = int(m_valid['bimodal_flag_dip'].mode()[0]) if 'bimodal_flag_dip' in m_valid.columns and len(m_valid) else 0

    onset_10 = m_valid['onset_doy_10'].mean() if 'onset_doy_10' in m_valid.columns and len(m_valid) else np.nan
    end_90   = m_valid['end_doy_90'].mean()   if 'end_doy_90'   in m_valid.columns and len(m_valid) else np.nan
    onset_15 = m_valid['onset_doy_15'].mean() if 'onset_doy_15' in m_valid.columns and len(m_valid) else np.nan
    end_85   = m_valid['end_doy_85'].mean()   if 'end_doy_85'   in m_valid.columns and len(m_valid) else np.nan
    len_10   = (end_90 - onset_10)            if pd.notna(onset_10) and pd.notna(end_90)  else np.nan
    len_15   = (end_85 - onset_15)            if pd.notna(onset_15) and pd.notna(end_85)  else np.nan

    # IQR window bounds
    if 'onset_doy_25' in m_valid.columns and 'end_doy_75' in m_valid.columns:
        iqr_onset = m_valid['onset_doy_25'].mean()
        iqr_end   = m_valid['end_doy_75'].mean()
    else:
        iqr_onset = peak_mean - iqr_len / 2 if pd.notna(peak_mean) and pd.notna(iqr_len) else np.nan
        iqr_end   = peak_mean + iqr_len / 2 if pd.notna(peak_mean) and pd.notna(iqr_len) else np.nan

    # ── PLOT ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 5))
    doys = np.arange(1, 366)

    # --- Full-height season window shades ---
    if pd.notna(onset_mean) and pd.notna(end_mean):
        ax.axvspan(onset_mean, end_mean, alpha=0.03, color=C_S95, zorder=0,
                   label='5–95% window')
    if pd.notna(iqr_onset) and pd.notna(iqr_end):
        ax.axvspan(iqr_onset, iqr_end, alpha=0.06, color=C_IQR, zorder=0,
                   label='IQR window')

    # --- Per-year thin lines ---
    for yr in years:
        yr_prop = daily_full[daily_full['year'] == yr]['prop'].values
        if len(yr_prop) >= 365:
            yr_smooth = pd.Series(yr_prop[:365]).rolling(15, center=True, min_periods=1).mean().values
            ax.plot(doys, yr_smooth, color=C_YR_LINE, linewidth=0.5, alpha=0.35, zorder=1)

    # --- Mean profile ---
    ax.plot(doys, smoothed, color=C_PROFILE, linewidth=2.2, zorder=3, label='Mean profile')

    # --- Vertical markers ---
    marker_kw = dict(zorder=4, linewidth=1.4)
    if pd.notna(onset_mean):
        ax.axvline(onset_mean,  color=C_ONSET,  label='Onset (5%)',  **marker_kw)
    if pd.notna(peak_mean):
        ax.axvline(peak_mean,   color=C_PEAK,   label='Peak',        **marker_kw)
    if pd.notna(median_mean):
        ax.axvline(median_mean, color=C_MEDIAN, label='Median',
                   linestyle='--', **marker_kw)
    if pd.notna(end_mean):
        ax.axvline(end_mean,    color=C_END,    label='End (95%)',   **marker_kw)

    # --- Bottom bars ---
    bar_h    = smoothed.max() * 0.04
    bar_step = smoothed.max() * 0.07

    season_bars = [
        (onset_mean, end_mean,  season_len, C_S95, '5–95%'),
        (onset_10,   end_90,    len_10,     C_10,  '10–90%'),
        (onset_15,   end_85,    len_15,     C_15,  '15–85%'),
        (iqr_onset,  iqr_end,   iqr_len,    C_IQR, 'IQR'),
    ]

    for rank, (b_onset, b_end, b_len, color, label) in enumerate(season_bars):
        bar_y = smoothed.max() * -0.07 - rank * bar_step
        if pd.notna(b_onset) and pd.notna(b_end):
            ax.barh(bar_y, width=b_end - b_onset, left=b_onset,
                    height=bar_h, color=color, alpha=0.85,
                    zorder=5, clip_on=False)
            len_str = f'{b_len:.0f}d' if pd.notna(b_len) else ''
            ax.text(b_onset, bar_y, f' {label}: {len_str}',
                    va='center', fontsize=7, color=color,
                    fontweight='bold', zorder=6, clip_on=False)

    # --- Annotation box (top-right) ---
    flag_str = '⚑ Flagged' if flag_eco == 1 else 'Clean'
    bc_str   = f'{bc_val:.3f}  {flag_str}' if pd.notna(bc_val) else f'—  {flag_str}'
    dip_str  = f'{dip_pval:.3f}  {"⚑" if flag_dip else "✓"}' if pd.notna(dip_pval) else '—'

    def fmt(val, decimals=1, suffix=''):
        return f'{val:.{decimals}f}{suffix}' if pd.notna(val) else '—'

    stats_lines = [
        ('Season length',  fmt(season_len,  1, ' d')),
        ('IQR season',     fmt(iqr_len,     1, ' d')),
        ('10% season',     fmt(len_10,      1, ' d')),
        ('15% season',     fmt(len_15,      1, ' d')),
        ('Active days',    fmt(active_days, 1, ' d')),
        ('Peak conc.',     fmt(peak_conc,   3)),
        ('Mean–med div.',  fmt(mm_div,      1, ' d')),
        ('Skewness',       fmt(skew_val,    2)),
        ('BC',             bc_str),
        ('Dip p-val',      dip_str),
        ('CV peak',        fmt(cv_eco,      3)),
        ('Profile corr.',  fmt(corr_eco,    3)),
    ]

    box_text = '\n'.join(f'{k:<16} {v}' for k, v in stats_lines)
    ax.text(0.78, 0.97, box_text,
            transform=ax.transAxes,
            fontsize=7.5, family='monospace',
            verticalalignment='top', horizontalalignment='left',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                      edgecolor='#cccccc', alpha=0.92),
            zorder=7,
            clip_on=True)

    # --- Axes ---
    ax.set_xlim(1, 365)
    ax.set_xticks(MONTH_STARTS)
    ax.set_xticklabels(MONTH_NAMES)
    ax.set_ylabel('Proportion of annual detections')
    ax.set_xlabel('Day of year')
    ax.grid(axis='y', alpha=0.2, linewidth=0.5)
    ax.spines[['top', 'right']].set_visible(False)

    # --- Title ---
    ax.set_title(
        f'{eco_name}  (ID: {eco_id})\n'
        f'Biome: {biome_name}  |  n = {len(years)} years',
        fontsize=10, fontweight='bold'
    )

    ax.legend(fontsize=7.5, ncol=4, frameon=False,
              loc='upper left', bbox_to_anchor=(0, 1.0))

    fig.subplots_adjust(bottom=0.22)
    plt.tight_layout()

    safe_name = eco_name.replace(' ', '_').replace('/', '_')
    save_path = os.path.join(profile_dir, f'{eco_id}_{safe_name}_climatology.png')
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight')
    plt.close()

print(f'Saved {len(eco_ids)} profile plots to {profile_dir}')

In [21]:
eco_rows = master_df[master_df['eco_id'] == 658]
print(eco_rows[['season_length', 'season_len_circ', 
                'peak_concentration', 'circ_R']].describe())

       season_length  season_len_circ  peak_concentration     circ_R
count      23.000000        23.000000           23.000000  23.000000
mean      255.565217       265.739130            0.488065   0.308126
std        38.396053        47.690496            0.166576   0.175576
min       176.000000       165.000000            0.232600   0.091600
25%       228.500000       228.500000            0.356500   0.182950
50%       250.000000       252.000000            0.509100   0.239300
75%       280.500000       311.000000            0.611500   0.414400
max       350.000000       339.000000            0.825900   0.762400


In [34]:
%%capture

# SECTION X: CIRCULAR PROFILE PLOTS + CLIMATE SPARKLINES ----------------------
# Two polar plots per ecoregion (standard vs circular method),
# plus climate context (ERA5 temperature and precipitation climatology).
#
# LAYOUT = 'A' : temp and precip in separate subplots below the polar plots
#                [ Standard polar ] [ Circular polar ]
#                [ Temp sparkline ] [ Precip sparkline ]
#
# LAYOUT = 'B' : single wide climate panel below both polar plots,
#                temperature (red, left axis) + precipitation (blue, right axis)
#                [ Standard polar ] [ Circular polar ]
#                [     Temp + Precip combined (full width)     ]

# ── USER CONTROLS ─────────────────────────────────────────────────────────────
all_eco_ids     = sorted(master_df['eco_id'].unique())
med_eco_ids = climatology_df['eco_id'].unique()
ECO_IDS_TO_PLOT = sorted(med_eco_ids)

SMOOTH_WINDOW = 15
CIRC_R_MIN    = 0.55
LAYOUT        = 'B'   # 'A' or 'B'

MONTH_STARTS = [1,  32,  60,  91, 121, 152, 182, 213, 244, 274, 305, 335]
MONTH_NAMES  = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

C_PROFILE = '#2c7bb6'
C_ARC     = 'black'
C_CENTER  = '#d62728'
C_TEMP    = '#d62728'
C_PREC    = '#2c7bb6'
C_SEASON  = '#ff7f0e'


# ── HELPERS ───────────────────────────────────────────────────────────────────

def circular_mean_doy(doys):
    doys = np.asarray(doys, dtype=float)
    doys = doys[~np.isnan(doys)]
    if len(doys) == 0:
        return np.nan
    angles   = 2 * np.pi * (doys - 1) / 365
    mean_sin = np.mean(np.sin(angles))
    mean_cos = np.mean(np.cos(angles))
    if abs(mean_sin) < 1e-10 and abs(mean_cos) < 1e-10:
        return np.nan
    mean_angle = np.arctan2(mean_sin, mean_cos)
    mean_doy   = (mean_angle * 365 / (2 * np.pi)) + 1
    if mean_doy < 1:
        mean_doy += 365
    return mean_doy


def shade_season(ax, onset, end):
    """Shade the fire season window on a linear DOY axis. Handles wrap-around."""
    if onset is None or end is None:
        return
    if np.isnan(onset) or np.isnan(end):
        return
    onset_i = int(round(onset))
    end_i   = int(round(end))
    if end_i >= onset_i:
        ax.axvspan(onset_i, end_i, alpha=0.18, color=C_SEASON,
                   label='Fire season window', zorder=2)
    else:
        ax.axvspan(onset_i, 365, alpha=0.18, color=C_SEASON, zorder=2)
        ax.axvspan(1,       end_i, alpha=0.18, color=C_SEASON,
                   label='Fire season window', zorder=2)


def style_clim_ax(ax):
    """Apply shared x-axis styling to a climate sparkline."""
    ax.set_xticks(MONTH_STARTS)
    ax.set_xticklabels(MONTH_NAMES, fontsize=7)
    ax.set_xlim(1, 365)
    ax.grid(True, alpha=0.2, linewidth=0.4)
    ax.tick_params(axis='y', labelsize=7)


def draw_sparklines_A(ax_temp, ax_prec, eco_clim, onset, end):
    """
    Layout A: temperature on ax_temp, precipitation on ax_prec.
    """
    if eco_clim is None or eco_clim.empty:
        for ax in [ax_temp, ax_prec]:
            ax.text(0.5, 0.5, 'No ERA5 data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=9, color='grey')
        return

    clim = eco_clim.sort_values('doy')
    doys = clim['doy'].values

    temp_smooth = (pd.Series(clim['temp_climatology'].values)
                   .rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean().values)
    prec_smooth = (pd.Series(clim['precip_climatology'].values)
                   .rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean().values)

    ax_temp.plot(doys, temp_smooth, color=C_TEMP, lw=1.5, zorder=3)
    ax_temp.fill_between(doys, temp_smooth, alpha=0.15, color=C_TEMP, zorder=3)
    shade_season(ax_temp, onset, end)
    ax_temp.set_ylabel('Temp (°C)', fontsize=8, color=C_TEMP)
    ax_temp.tick_params(axis='y', labelcolor=C_TEMP)
    ax_temp.set_title('Temperature climatology', fontsize=8, pad=4)
    style_clim_ax(ax_temp)
    ax_temp.legend(fontsize=7, loc='upper right', framealpha=0.8)

    ax_prec.fill_between(doys, prec_smooth, alpha=0.35, color=C_PREC, zorder=3)
    ax_prec.plot(doys, prec_smooth, color=C_PREC, lw=1.5, zorder=3)
    shade_season(ax_prec, onset, end)
    ax_prec.set_ylabel('Precip (mm/day)', fontsize=8, color=C_PREC)
    ax_prec.tick_params(axis='y', labelcolor=C_PREC)
    ax_prec.set_title('Precipitation climatology', fontsize=8, pad=4)
    style_clim_ax(ax_prec)
    ax_prec.legend(fontsize=7, loc='upper right', framealpha=0.8)


def draw_climate_panel_B(ax, eco_clim, onset, end):
    """
    Layout B: temperature (red, left axis) and precipitation (blue, right axis)
    on a single wide panel.
    """
    if eco_clim is None or eco_clim.empty:
        ax.text(0.5, 0.5, 'No ERA5 data', transform=ax.transAxes,
                ha='center', va='center', fontsize=9, color='grey')
        return

    clim = eco_clim.sort_values('doy')
    doys = clim['doy'].values

    temp_smooth = (pd.Series(clim['temp_climatology'].values)
                   .rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean().values)
    prec_smooth = (pd.Series(clim['precip_climatology'].values)
                   .rolling(SMOOTH_WINDOW, center=True, min_periods=1).mean().values)

    ax.plot(doys, temp_smooth, color=C_TEMP, lw=1.8, zorder=3,
            label='Temperature (°C)')
    ax.fill_between(doys, temp_smooth, alpha=0.12, color=C_TEMP, zorder=3)
    ax.set_ylabel('Temperature (°C)', fontsize=8, color=C_TEMP)
    ax.tick_params(axis='y', labelcolor=C_TEMP, labelsize=7)

    ax2 = ax.twinx()
    ax2.fill_between(doys, prec_smooth, alpha=0.30, color=C_PREC, zorder=2)
    ax2.plot(doys, prec_smooth, color=C_PREC, lw=1.5, zorder=2,
             label='Precipitation (mm/day)')
    ax2.set_ylabel('Precipitation (mm/day)', fontsize=8, color=C_PREC)
    ax2.tick_params(axis='y', labelcolor=C_PREC, labelsize=7)

    shade_season(ax, onset, end)

    style_clim_ax(ax)
    ax.set_title('ERA5 climatology  (temperature & precipitation)',
                 fontsize=8, pad=4)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2,
              fontsize=7, loc='upper right', framealpha=0.85)


def draw_circular_profile(ax, smoothed, onset, end, circ_mean,
                          season_len, method_label, len_other=None,
                          stats=None):
    """
    Draw one polar fire-season profile with optional stats annotation box.
    """
    doys_full   = np.arange(1, 366)
    angles_full = (2 * np.pi * (doys_full - 1) / 365)
    vals        = smoothed.values
    ymax        = vals.max()

    # 1. Profile fill and line
    angles_plot = np.append(angles_full, angles_full[0])
    vals_plot   = np.append(vals, vals[0])
    ax.fill(angles_plot, vals_plot, color=C_PROFILE, alpha=0.25, zorder=3)
    ax.plot(angles_plot, vals_plot, color=C_PROFILE, linewidth=2.0, zorder=4)

    arc_radius  = ymax * 1.18
    onset_valid = (onset is not None) and (not np.isnan(onset))
    end_valid   = (end   is not None) and (not np.isnan(end))

    # 2. Arc between onset and end
    if onset_valid and end_valid:
        onset_angle = (2 * np.pi * (onset - 1) / 365)
        end_angle   = (2 * np.pi * (end   - 1) / 365)
        if end_angle >= onset_angle:
            arc_angles = np.linspace(onset_angle, end_angle, 300)
        else:
            arc_angles = np.linspace(onset_angle, end_angle + 2 * np.pi, 300)
            arc_angles = arc_angles % (2 * np.pi)
        ax.plot(arc_angles, np.full_like(arc_angles, arc_radius),
                color=C_ARC, linewidth=5, zorder=6, solid_capstyle='round')
        tick_inner = arc_radius * 0.88
        tick_outer = arc_radius * 1.12
        for angle in [onset_angle, end_angle % (2 * np.pi)]:
            ax.plot([angle, angle], [tick_inner, tick_outer],
                    color=C_ARC, linewidth=2.0, zorder=7)

    # 3. Circular mean triangle
    center_valid = (circ_mean is not None) and (not np.isnan(circ_mean))
    if center_valid:
        circ_angle = (2 * np.pi * (circ_mean - 1) / 365)
        ax.plot(circ_angle, ymax * 1.05,
                marker='v', color=C_CENTER, markersize=9,
                zorder=8, clip_on=False,
                label=f'Center DOY {int(round(circ_mean))}')

    # 4. Month labels
    label_radius = arc_radius * 1.18
    for doy, name in zip(MONTH_STARTS, MONTH_NAMES):
        ang = (2 * np.pi * (doy - 1) / 365)
        ax.text(ang, label_radius, name,
                ha='center', va='center', fontsize=8,
                color='#333333', fontweight='bold')

    # 5. Center season length annotation
    if (season_len is not None) and (not np.isnan(season_len)):
        len_str = f'{int(round(season_len))} d'
    else:
        len_str = '—'

    if (len_other is not None) and (not np.isnan(len_other)) and \
       (season_len is not None) and (not np.isnan(season_len)):
        delta     = int(round(season_len)) - int(round(len_other))
        sign      = '+' if delta > 0 else ''
        delta_str = f'\nΔ {sign}{delta} d vs other'
    else:
        delta_str = ''

    ax.text(0, 0, f'{len_str}{delta_str}',
            ha='center', va='center',
            fontsize=10, fontweight='bold', color='#222222', zorder=9)

    # 6. Polar settings
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_xticks([])
    ax.set_yticklabels([])
    ax.set_ylim(0, arc_radius * 1.35)
    ax.grid(True, alpha=0.15, linewidth=0.5)
    ax.set_title(method_label, fontsize=10, fontweight='bold', pad=14)

    # 7. Stats annotation box
    if stats is not None:

        def fmt(v, decimals=0):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                return '—'
            return f'{round(v, decimals)}'

        left_lines = [
            '── Season length ──',
            f'  mean    {fmt(stats.get("mean"))} d',
            f'  median  {fmt(stats.get("median"))} d',
            f'  std     {fmt(stats.get("std"))} d',
            f'  IQR     {fmt(stats.get("iqr"))} d',
            f'  min     {fmt(stats.get("min"))} d',
            f'  max     {fmt(stats.get("max"))} d',
            '── Peak ───────────',
            f'  DOY     {fmt(stats.get("peak_doy_mean"))} d',
            f'  ± std   {fmt(stats.get("peak_doy_std"))} d',
            f'  conc    {fmt(stats.get("peak_conc", 0) * 100, decimals=1)} %',
        ]

        right_lines = [
            '── Activity ───────',
            f'  mean det  {fmt(stats.get("mean_dets"))}/yr',
            f'  active    {stats.get("n_active","—")}/{stats.get("n_years","—")} yrs',
            f'  coherent  {stats.get("n_coherent","—")}/{stats.get("n_years","—")} yrs',
            '── Quality ────────',
            f'  CV peak   {fmt(stats.get("cv_peak"), decimals=2)}',
            f'  profile   {fmt(stats.get("profile_corr"), decimals=2)}',
        ]

        shared_kw = dict(
            transform   = ax.transAxes,
            fontsize    = 8.5,
            family      = 'monospace',
            va          = 'top',
            linespacing = 1.5,
            zorder      = 10,
        )

        # Left block
        ax.text(0.18, -0.12, '\n'.join(left_lines),
                ha='left',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                          edgecolor='#cccccc', alpha=0.9),
                **shared_kw)

        # Right block
        ax.text(0.82, -0.12, '\n'.join(right_lines),
                ha='right',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                          edgecolor='#cccccc', alpha=0.9),
                **shared_kw)

    # 8. Legend
    if center_valid:
        ax.legend(fontsize=8, loc='lower left',
                  bbox_to_anchor=(-0.12, -0.12),
                  frameon=False)


# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
for eco_id in ECO_IDS_TO_PLOT:

    eco_rows   = master_df[master_df['eco_id'] == eco_id]
    daily_rows = daily_df[daily_df['eco_id'] == eco_id].copy()

    if eco_rows.empty or daily_rows.empty:
        continue

    eco_name   = eco_rows['eco_name'].iloc[0]
    biome_name = eco_rows['biome_name'].iloc[0]
    n_years    = eco_rows['year'].nunique()

    # ERA5 climatology for this ecoregion
    if climatology_df is not None:
        eco_clim = climatology_df[climatology_df['eco_id'] == eco_id].copy()
        eco_clim = eco_clim if not eco_clim.empty else None
    else:
        eco_clim = None

    # Circular stats
    circ_R     = eco_rows['circ_R'].mean()
    circ_rot   = int(eco_rows['circ_rotated'].max())
    circ_mean  = circular_mean_doy(eco_rows['circ_mean_doy'])
    is_diffuse = np.isnan(circ_R) or circ_R < CIRC_R_MIN

    # DOY means — circular
    onset_std  = circular_mean_doy(eco_rows['onset_doy'])
    end_std    = circular_mean_doy(eco_rows['end_doy'])
    onset_circ = circular_mean_doy(eco_rows['onset_doy_circ'])
    end_circ   = circular_mean_doy(eco_rows['end_doy_circ'])

    # Length means — arithmetic
    len_std  = eco_rows['season_length'].mean()
    len_circ = eco_rows['season_len_circ'].mean()

    # Tier 1 stats — standard method
    std_mean   = eco_rows['season_length'].mean()
    std_median = eco_rows['season_length'].median()
    std_std    = eco_rows['season_length'].std()
    std_min    = eco_rows['season_length'].min()
    std_max    = eco_rows['season_length'].max()
    std_iqr    = eco_rows['season_length'].quantile(0.75) - \
                 eco_rows['season_length'].quantile(0.25)

    # Tier 1 stats — circular method
    circ_mean_len   = eco_rows['season_len_circ'].mean()
    circ_median_len = eco_rows['season_len_circ'].median()
    circ_std_len    = eco_rows['season_len_circ'].std()
    circ_min_len    = eco_rows['season_len_circ'].min()
    circ_max_len    = eco_rows['season_len_circ'].max()
    circ_iqr_len    = eco_rows['season_len_circ'].quantile(0.75) - \
                      eco_rows['season_len_circ'].quantile(0.25)

    # Shared stats
    peak_conc     = eco_rows['peak_concentration'].mean()
    n_coherent    = int((eco_rows['circ_R'] >= CIRC_R_MIN).sum())
    peak_doy_mean = circular_mean_doy(eco_rows['peak_doy'])
    peak_doy_std  = eco_rows['peak_doy'].std()
    mean_dets     = eco_rows['n_detections'].mean()
    n_active      = int(eco_rows['n_detections'].notna().sum())
    cv_peak       = eco_rows['cv_peak_doy'].mean()
    profile_corr  = eco_rows['mean_profile_corr'].mean()

    # Season window for climate shading — prefer circular, fall back to standard
    onset_for_clim = onset_circ if (not np.isnan(onset_circ)) else onset_std
    end_for_clim   = end_circ   if (not np.isnan(end_circ))   else end_std

    # Smoothed mean annual fire profile
    daily_rows['norm'] = daily_rows.groupby('year')['n_detections'].transform(
        lambda x: x / x.sum() if x.sum() > 0 else x
    )
    mean_profile = (
        daily_rows.groupby('doy')['norm']
        .mean()
        .reindex(range(1, 366), fill_value=0)
    )
    smoothed = mean_profile.rolling(SMOOTH_WINDOW, center=True,
                                    min_periods=1).mean()

    # Coherence tag
    if is_diffuse:
        coherence_tag = f'✗ Diffuse (R<{CIRC_R_MIN})'
    elif circ_rot:
        coherence_tag = '⚑ Near boundary'
    else:
        coherence_tag = '✓ Standard'

    # ── Figure layout ─────────────────────────────────────────────────────────
    if LAYOUT == 'A':
        fig = plt.figure(figsize=(18, 14))
        gs  = fig.add_gridspec(2, 2, height_ratios=[3, 1.5],
                               hspace=0.65, wspace=0.5)
        ax_std  = fig.add_subplot(gs[0, 0], projection='polar')
        ax_circ = fig.add_subplot(gs[0, 1], projection='polar')
        ax_temp = fig.add_subplot(gs[1, 0])
        ax_prec = fig.add_subplot(gs[1, 1])
    else:
        fig = plt.figure(figsize=(18, 14))
        gs  = fig.add_gridspec(2, 2, height_ratios=[3, 1.5],
                               hspace=0.65, wspace=0.5)
        ax_std     = fig.add_subplot(gs[0, 0], projection='polar')
        ax_circ    = fig.add_subplot(gs[0, 1], projection='polar')
        ax_climate = fig.add_subplot(gs[1, :])

    fig.suptitle(
        f'{eco_name}  (ID: {eco_id})\n'
        f'Biome: {biome_name}  |  n = {n_years} years  |  '
        f'R = {circ_R:.3f}  |  {coherence_tag}',
        fontsize=11, fontweight='bold', y=1.02
    )

    # ── Polar plots ───────────────────────────────────────────────────────────
    draw_circular_profile(
        ax=ax_std, smoothed=smoothed, onset=onset_std, end=end_std,
        circ_mean=circ_mean, season_len=len_std,
        method_label='Standard method', len_other=len_circ,
        stats=dict(
            mean=std_mean, median=std_median, std=std_std,
            iqr=std_iqr, min=std_min, max=std_max,
            peak_doy_mean=peak_doy_mean, peak_doy_std=peak_doy_std,
            peak_conc=peak_conc, mean_dets=mean_dets,
            n_active=n_active, n_coherent=n_coherent, n_years=n_years,
            cv_peak=cv_peak, profile_corr=profile_corr,
        ),
    )
    draw_circular_profile(
        ax=ax_circ, smoothed=smoothed, onset=onset_circ, end=end_circ,
        circ_mean=circ_mean, season_len=len_circ,
        method_label='Circular method', len_other=len_std,
        stats=dict(
            mean=circ_mean_len, median=circ_median_len, std=circ_std_len,
            iqr=circ_iqr_len, min=circ_min_len, max=circ_max_len,
            peak_doy_mean=peak_doy_mean, peak_doy_std=peak_doy_std,
            peak_conc=peak_conc, mean_dets=mean_dets,
            n_active=n_active, n_coherent=n_coherent, n_years=n_years,
            cv_peak=cv_peak, profile_corr=profile_corr,
        ),
    )

    # ── Climate panels ────────────────────────────────────────────────────────
    if LAYOUT == 'A':
        draw_sparklines_A(ax_temp, ax_prec, eco_clim,
                          onset_for_clim, end_for_clim)
    else:
        draw_climate_panel_B(ax_climate, eco_clim,
                             onset_for_clim, end_for_clim)

    # ── Save ──────────────────────────────────────────────────────────────────
    safe_name = eco_name.replace(' ', '_').replace('/', '_')
    save_path = os.path.join(profile_dir,
                             f'{eco_id}_{safe_name}_circular_comparison.png')
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches='tight')
    plt.close()

## Section 3: Bimodality Flag Inspection

In [ ]:
metrics_df = pd.read_csv(os.path.join(output_dir, '_all_metrics.csv'))
daily_df   = pd.read_csv(os.path.join(output_dir, '_all_daily_counts.csv'))

In [ ]:
# PLOT: ECOREGION BIMODAL FLAG MAP — CONSENSUS STRENGTH (global) ------------------------------
# Draws all ecoregion polygons from eco_geometries.json, colored by bimodal_consensus_eco:
#   2 = strong  (both BC and dip agree) → red
#   1 = weak    (one method only)       → orange
#   0 = clean   (neither)               → blue
#  -1 = no data                         → grey
#
# Uses PatchCollection for fast rendering at global scale (~846 ecoregions).
# Two-pass z-ordering: grey base first, colored data ecoregions on top.
#
# Requires: quality_df, geo_lookup, extract_rings (all from earlier cells).

# ── Helper ────────────────────────────────────────────────────────────────────
def extract_rings(geometry):
    """Return a list of exterior rings (each a list of [lon, lat] pairs)."""
    geo_type = geometry.get('type', '')
    coords   = geometry.get('coordinates', [])
    if geo_type == 'Polygon':
        return [coords[0]] if coords else []
    elif geo_type == 'MultiPolygon':
        return [poly[0] for poly in coords if poly]
    return []

# ── Consensus lookup ──────────────────────────────────────────────────────────
consensus_lookup = (
    quality_df.set_index('eco_id')['bimodal_consensus_eco']
    .to_dict()
)

# ── Colors and labels ─────────────────────────────────────────────────────────
C_CONSENSUS = {
     2: '#d62728',   # red    — both methods agree: bimodal
     1: '#ff7f0e',   # orange — one method only
     0: '#1f77b4',   # blue   — clean
    -1: '#cccccc',   # light grey — no data
}
L_CONSENSUS = {
     2: 'Both flagged (strong)',
     1: 'One method only (weak)',
     0: 'Clean',
    -1: 'No data',
}

# ── Collect patches into bins ─────────────────────────────────────────────────
patch_bins = {k: [] for k in [-1, 0, 1, 2]}

for eco_id, geometry in geo_lookup.items():
    consensus_val = consensus_lookup.get(eco_id, -1)
    for ring in extract_rings(geometry):
        if len(ring) < 3:
            continue
        xy = [(pt[0], pt[1]) for pt in ring]
        patch_bins[consensus_val].append(Polygon(xy, closed=True))

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 10))
ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('white')

# Render order: no-data (bottom) → clean → weak → strong (top)
render_order = [
    (-1, 1),   # zorder
    ( 0, 2),
    ( 1, 3),
    ( 2, 4),
]

for consensus_val, zorder in render_order:
    patches = patch_bins[consensus_val]
    if not patches:
        continue
    pc = PatchCollection(
        patches,
        facecolor = C_CONSENSUS[consensus_val],
        edgecolor = 'white',
        linewidth = 0.15,
        alpha     = 0.85,
        zorder    = zorder,
    )
    ax.add_collection(pc)

# ── Axes and labels ───────────────────────────────────────────────────────────
ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)          # trim Antarctic ice / empty Arctic
ax.set_aspect('equal')
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude',  fontsize=10)
ax.set_title(
    'Ecoregion Bimodal Flag — Consensus Strength',
    fontsize=14, fontweight='bold'
)


# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(color=C_CONSENSUS[k], label=L_CONSENSUS[k])
    for k in [2, 1, 0, -1]
]
ax.legend(
    handles  = legend_handles,
    loc      = 'lower left',
    fontsize = 10,
    framealpha = 0.9,
)

# ── Gridlines ─────────────────────────────────────────────────────────────────
ax.grid(alpha=0.15, linewidth=0.4)

# ── Counts annotation ─────────────────────────────────────────────────────────
counts_text = '\n'.join(
    f'{L_CONSENSUS[k]}: {len(patch_bins[k])} polygons'
    for k in [2, 1, 0, -1]
)
ax.text(
    0.99, 0.01, counts_text,
    transform=ax.transAxes,
    fontsize=8, family='monospace',
    va='bottom', ha='right',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor='#cccccc', alpha=0.9),
)

plt.tight_layout()
plt.savefig(
    os.path.join(plots_dir, '99b_bimodal_flag_map_consensus.png'),
    dpi=FIG_DPI, bbox_inches='tight',
)
plt.show()

# ── Summary print ─────────────────────────────────────────────────────────────
n_eco_per_class = quality_df['bimodal_consensus_eco'].value_counts().sort_index()
print('Ecoregion counts by consensus class:')
for val, count in n_eco_per_class.items():
    print(f'  {int(val)} ({L_CONSENSUS.get(int(val), "?"):<28s}): {count}')
print(f'  No data (not in quality_df): {len(geo_lookup) - len(quality_df)}')


## Section 5: Global-Scale Diagnostics

In [ ]:
# 5.1  HEMISPHERE SEASONALITY CHECK -------------------------------------------------------
# Flags ecoregion-years where onset_doy > end_doy (year-wrapping fire season).
# Expected in Southern Hemisphere ecoregions with Dec–Feb fire peaks.
# If widespread, the Jan–Dec calendar year assumption needs revisiting.

# --- Per ecoregion-year flag ---
master_df['wrap_flag'] = (master_df['onset_doy'] > master_df['end_doy']).astype(int)

# --- Ecoregion-level summary: fraction of years with wrapping ---
wrap_summary = (
    master_df.groupby('eco_id')
    .agg(
        eco_name    = ('eco_name', 'first'),
        biome_name  = ('biome_name', 'first'),
        n_years     = ('year', 'count'),
        n_wrap      = ('wrap_flag', 'sum'),
    )
    .reset_index()
)
wrap_summary['frac_wrap'] = round(wrap_summary['n_wrap'] / wrap_summary['n_years'], 3)

# Ecoregions where >30% of years wrap
chronic_wrap = wrap_summary[wrap_summary['frac_wrap'] > 0.30].sort_values('frac_wrap', ascending=False)

print(f'Total ecoregion-years with onset > end: {master_df["wrap_flag"].sum()} '
      f'/ {len(master_df)} ({master_df["wrap_flag"].mean()*100:.1f}%)')
print(f'Ecoregions with >30% year-wrapping: {len(chronic_wrap)} / {len(wrap_summary)}')
print()

if len(chronic_wrap) > 0:
    print(chronic_wrap[['eco_id', 'eco_name', 'biome_name', 'n_years', 'n_wrap', 'frac_wrap']]
          .head(20).to_string(index=False))
else:
    print('No chronic year-wrapping ecoregions found.')


In [ ]:
# 5.1b  MAP: YEAR-WRAPPING ECOREGIONS -----------------------------------------------------
# Colors ecoregions by fraction of years with onset > end.
# Expect clusters in southern Africa, South America, Australia.

import matplotlib.colors as mcolors

wrap_lookup = wrap_summary.set_index('eco_id')['frac_wrap'].to_dict()

# Custom colormap: white (0) → yellow → red (1)
cmap_wrap = mcolors.LinearSegmentedColormap.from_list(
    'wrap', ['#f0f0f0', '#fee08b', '#e6550d', '#a50026']
)
norm_wrap = mcolors.Normalize(vmin=0, vmax=1)

patches_wrap  = []
colors_wrap   = []
patches_nodata = []

for eco_id, geometry in geo_lookup.items():
    frac = wrap_lookup.get(eco_id, None)
    for ring in extract_rings(geometry):
        if len(ring) < 3:
            continue
        xy = [(pt[0], pt[1]) for pt in ring]
        if frac is not None:
            patches_wrap.append(Polygon(xy, closed=True))
            colors_wrap.append(cmap_wrap(norm_wrap(frac)))
        else:
            patches_nodata.append(Polygon(xy, closed=True))

fig, ax = plt.subplots(figsize=(20, 10))
ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('white')

if patches_nodata:
    pc_nd = PatchCollection(patches_nodata, facecolor='#e0e0e0',
                            edgecolor='white', linewidth=0.1, alpha=0.6, zorder=1)
    ax.add_collection(pc_nd)

if patches_wrap:
    pc_w = PatchCollection(patches_wrap, facecolor=colors_wrap,
                           edgecolor='white', linewidth=0.1, alpha=0.85, zorder=2)
    ax.add_collection(pc_w)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap_wrap, norm=norm_wrap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Fraction of years with onset > end', fontsize=10)

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)
ax.set_aspect('equal')
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude', fontsize=10)
ax.set_title('5.1  Year-Wrapping Fire Seasons (onset DOY > end DOY)',
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.15, linewidth=0.4)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '51_year_wrap_map.png'),
            dpi=FIG_DPI, bbox_inches='tight')
plt.show()


In [ ]:
# 5.2  FIRE ACTIVITY COVERAGE — VALID YEARS PER ECOREGION ---------------------------------
# How many ecoregions survived the MIN_DETECTIONS filter, and for how many years?
# Ecoregions with 0 valid years are in geo_lookup but absent from master_df.

# Build coverage from master_df (only ecoregions that passed threshold)
coverage = (
    master_df.groupby('eco_id')
    .agg(
        eco_name       = ('eco_name', 'first'),
        biome_name     = ('biome_name', 'first'),
        n_years_valid  = ('year', 'count'),
        mean_detections= ('n_detections', 'mean'),
    )
    .reset_index()
)

# Add ecoregions with zero valid years
all_eco_ids = set(geo_lookup.keys())
in_master   = set(coverage['eco_id'])
missing_ids = all_eco_ids - in_master

missing_rows = []
for eco_id in missing_ids:
    # Try to get name from geo_data
    match = [r for r in geo_data if r['eco_id'] == eco_id]
    name  = match[0]['eco_name']  if match else ''
    biome = match[0]['biome_name'] if match else ''
    missing_rows.append({
        'eco_id': eco_id, 'eco_name': name, 'biome_name': biome,
        'n_years_valid': 0, 'mean_detections': 0,
    })

coverage_full = pd.concat([coverage, pd.DataFrame(missing_rows)], ignore_index=True)

# Bin years
bins   = [0, 1, 6, 11, 16, 24]
labels = ['0', '1–5', '6–10', '11–15', '16–23']
coverage_full['year_bin'] = pd.cut(
    coverage_full['n_years_valid'], bins=bins, labels=labels,
    right=False, include_lowest=True
)

print('Ecoregion coverage by valid-year bins:')
bin_counts = coverage_full['year_bin'].value_counts().sort_index()
for b, c in bin_counts.items():
    print(f'  {str(b):<8s}: {c:>4d}  ({c/len(coverage_full)*100:.1f}%)')
print(f'  Total   : {len(coverage_full):>4d}')
print()

# Biome-level dropout summary
biome_cov = (
    coverage_full.groupby('biome_name')
    .agg(
        n_eco     = ('eco_id', 'count'),
        n_zero    = ('n_years_valid', lambda x: (x == 0).sum()),
        mean_yrs  = ('n_years_valid', 'mean'),
    )
    .reset_index()
)
biome_cov['pct_dropout'] = round(biome_cov['n_zero'] / biome_cov['n_eco'] * 100, 1)
biome_cov = biome_cov.sort_values('pct_dropout', ascending=False)

print('Biome-level dropout (ecoregions with 0 valid years):')
print(biome_cov[['biome_name', 'n_eco', 'n_zero', 'pct_dropout', 'mean_yrs']]
      .to_string(index=False))


In [ ]:
# 5.2b  MAP: VALID YEARS PER ECOREGION ----------------------------------------------------

coverage_lookup = coverage_full.set_index('eco_id')['n_years_valid'].to_dict()

# Discrete colormap: 5 bins
bin_colors = {
    '0'    : '#d9d9d9',
    '1–5'  : '#fdae61',
    '6–10' : '#fee08b',
    '11–15': '#a6d96a',
    '16–23': '#1a9850',
}

def year_to_bin(n):
    if n == 0:    return '0'
    if n <= 5:    return '1–5'
    if n <= 10:   return '6–10'
    if n <= 15:   return '11–15'
    return '16–23'

patch_bins_cov = {k: [] for k in bin_colors}

for eco_id, geometry in geo_lookup.items():
    n_yrs = coverage_lookup.get(eco_id, 0)
    b     = year_to_bin(n_yrs)
    for ring in extract_rings(geometry):
        if len(ring) < 3:
            continue
        xy = [(pt[0], pt[1]) for pt in ring]
        patch_bins_cov[b].append(Polygon(xy, closed=True))

fig, ax = plt.subplots(figsize=(20, 10))
ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('white')

render_cov = ['0', '1–5', '6–10', '11–15', '16–23']
for zorder, b in enumerate(render_cov, start=1):
    if not patch_bins_cov[b]:
        continue
    pc = PatchCollection(
        patch_bins_cov[b],
        facecolor=bin_colors[b], edgecolor='white',
        linewidth=0.1, alpha=0.85, zorder=zorder,
    )
    ax.add_collection(pc)

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)
ax.set_aspect('equal')
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude', fontsize=10)
ax.set_title('5.2  Fire Activity Coverage — Valid Years per Ecoregion',
             fontsize=14, fontweight='bold')

legend_handles = [
    mpatches.Patch(color=bin_colors[b], label=f'{b} years ({len(patch_bins_cov[b])} polygons)')
    for b in render_cov
]
ax.legend(handles=legend_handles, loc='lower left', fontsize=9, framealpha=0.9)
ax.grid(alpha=0.15, linewidth=0.4)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '52_coverage_map.png'),
            dpi=FIG_DPI, bbox_inches='tight')
plt.show()


In [ ]:
# 5.3  TEMPORAL COMPLETENESS — FRACTION OF ECOREGIONS WITH VALID DATA PER YEAR ------------
# Line plot: what fraction of all fire-active ecoregions have valid metrics each year?
# Also: per-biome heatmap of mean detections per year.

YEARS = list(range(2003, 2026))

# Global line: fraction of ecoregions with data per year
n_eco_total = master_df['eco_id'].nunique()
yearly_cov  = (
    master_df.groupby('year')['eco_id']
    .nunique()
    .reindex(YEARS, fill_value=0)
)
frac_yearly = yearly_cov / n_eco_total

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [1, 2]})

# --- Top panel: line plot ---
ax = axes[0]
ax.plot(YEARS, frac_yearly.values, 'o-', color='#1f77b4', linewidth=1.5, markersize=4)
ax.set_ylabel('Fraction of ecoregions\nwith valid data')
ax.set_ylim(0, 1.05)
ax.set_title('5.3  Temporal Completeness', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# --- Bottom panel: biome × year heatmap (mean n_detections) ---
ax = axes[1]

biome_year = (
    master_df.groupby(['biome_name', 'year'])['eco_id']
    .nunique()
    .reset_index()
    .rename(columns={'eco_id': 'n_eco'})
)

# Total ecoregions per biome (from coverage_full)
biome_totals = coverage_full.groupby('biome_name')['eco_id'].nunique()
biome_year = biome_year.merge(biome_totals.rename('n_eco_total'), on='biome_name')
biome_year['frac'] = biome_year['n_eco'] / biome_year['n_eco_total']

pivot = biome_year.pivot_table(
    index='biome_name', columns='year', values='frac', fill_value=0
)
# Sort biomes by mean coverage
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=True).index]

im = ax.imshow(pivot.values, aspect='auto', cmap='YlGn', vmin=0, vmax=1)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index, fontsize=7)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=7, rotation=45, ha='right')
ax.set_xlabel('Year')
ax.set_ylabel('Biome')

cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label('Fraction of ecoregions with valid data', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '53_temporal_completeness.png'),
            dpi=FIG_DPI, bbox_inches='tight')
plt.show()


In [ ]:
# 5.4  BIOME-LEVEL METRIC DISTRIBUTIONS ---------------------------------------------------
# Boxplots of key timing metrics grouped by biome.
# Fast way to spot outlier ecoregions within a biome.

# Use ecoregion-level means (one value per ecoregion)
eco_means = (
    master_df.groupby(['eco_id', 'biome_name'])
    .agg(
        onset_doy     = ('onset_doy', 'mean'),
        peak_doy      = ('peak_doy', 'mean'),
        end_doy       = ('end_doy', 'mean'),
        season_length = ('season_length', 'mean'),
    )
    .reset_index()
)

metrics_to_plot = ['onset_doy', 'peak_doy', 'end_doy', 'season_length']
titles          = ['Onset DOY', 'Peak DOY', 'End DOY', 'Season Length (days)']

# Sort biomes by median peak_doy for consistent ordering
biome_order = (
    eco_means.groupby('biome_name')['peak_doy']
    .median()
    .sort_values()
    .index.tolist()
)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    ax = axes[idx]

    # Prepare data in biome order
    data_by_biome = []
    labels_biome  = []
    for biome in biome_order:
        vals = eco_means.loc[eco_means['biome_name'] == biome, metric].dropna()
        if len(vals) > 0:
            data_by_biome.append(vals.values)
            labels_biome.append(f'{biome} (n={len(vals)})')

    bp = ax.boxplot(
        data_by_biome, vert=False, patch_artist=True,
        boxprops=dict(facecolor='#aec7e8', linewidth=0.5),
        medianprops=dict(color='#d62728', linewidth=1.2),
        whiskerprops=dict(linewidth=0.5),
        capprops=dict(linewidth=0.5),
        flierprops=dict(marker='.', markersize=3, alpha=0.5),
    )

    ax.set_yticklabels(labels_biome, fontsize=7)
    ax.set_xlabel(title, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('5.4  Biome-Level Metric Distributions (ecoregion means)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '54_biome_metric_distributions.png'),
            dpi=FIG_DPI, bbox_inches='tight')
plt.show()


In [ ]:
# 5.5  SPATIAL COHERENCE — MEAN PEAK DOY MAP ----------------------------------------------
# Choropleth of mean peak DOY per ecoregion.
# Pattern should be geographically smooth: Mediterranean-climate ecoregions
# in California, Chile, South Africa, Australia should all show summer peaks.
# Salt-and-pepper noise would indicate pipeline issues.

import matplotlib.colors as mcolors

mean_peak = (
    master_df.groupby('eco_id')['peak_doy']
    .mean()
    .to_dict()
)

# Circular-ish colormap for DOY (wraps at year boundary)
# HSV works well: hue cycles through the year
cmap_doy = plt.cm.hsv
norm_doy = mcolors.Normalize(vmin=1, vmax=365)

patches_doy  = []
colors_doy   = []
patches_nd   = []

for eco_id, geometry in geo_lookup.items():
    peak = mean_peak.get(eco_id, None)
    for ring in extract_rings(geometry):
        if len(ring) < 3:
            continue
        xy = [(pt[0], pt[1]) for pt in ring]
        if peak is not None:
            patches_doy.append(Polygon(xy, closed=True))
            colors_doy.append(cmap_doy(norm_doy(peak)))
        else:
            patches_nd.append(Polygon(xy, closed=True))

fig, ax = plt.subplots(figsize=(20, 10))
ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('white')

if patches_nd:
    pc = PatchCollection(patches_nd, facecolor='#e0e0e0',
                         edgecolor='white', linewidth=0.1, alpha=0.5, zorder=1)
    ax.add_collection(pc)

if patches_doy:
    pc = PatchCollection(patches_doy, facecolor=colors_doy,
                         edgecolor='white', linewidth=0.1, alpha=0.85, zorder=2)
    ax.add_collection(pc)

# Colorbar with month labels
sm = plt.cm.ScalarMappable(cmap=cmap_doy, norm=norm_doy)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Mean Peak DOY', fontsize=10)
cbar_ticks = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
cbar_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
cbar.set_ticks(cbar_ticks)
cbar.set_ticklabels(cbar_labels)

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)
ax.set_aspect('equal')
ax.set_xlabel('Longitude', fontsize=10)
ax.set_ylabel('Latitude', fontsize=10)
ax.set_title('5.5  Spatial Coherence — Mean Peak DOY per Ecoregion',
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.15, linewidth=0.4)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '55_mean_peak_doy_map.png'),
            dpi=FIG_DPI, bbox_inches='tight')
plt.show()


In [ ]:
# 5.6  DETECTION COUNT VS METRIC STABILITY ------------------------------------------------
# Scatter: mean annual n_detections vs cv_peak_doy per ecoregion.
# Low-detection ecoregions likely have noisier metrics.
# If strong negative relationship → consider stricter MIN_DETECTIONS or quality tiers.

stab = (
    master_df.groupby('eco_id')
    .agg(
        mean_det   = ('n_detections', 'mean'),
        biome_name = ('biome_name', 'first'),
    )
    .reset_index()
    .merge(quality_df[['eco_id', 'cv_peak_doy']], on='eco_id', how='left')
)

stab = stab.dropna(subset=['cv_peak_doy'])

fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    stab['mean_det'], stab['cv_peak_doy'],
    s=12, alpha=0.5, c='#1f77b4', edgecolors='none',
)

ax.set_xscale('log')
ax.set_xlabel('Mean annual fire detections (log scale)', fontsize=11)
ax.set_ylabel('CV of peak DOY', fontsize=11)
ax.set_title('5.6  Detection Count vs. Peak Timing Stability',
             fontsize=13, fontweight='bold')
ax.grid(alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)

# Add Spearman correlation
from scipy.stats import spearmanr
rho, pval = spearmanr(stab['mean_det'], stab['cv_peak_doy'])
ax.text(0.02, 0.97, f'Spearman ρ = {rho:.3f} (p = {pval:.2e})',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='#cccccc', alpha=0.9))

# Threshold line at MIN_DETECTIONS
ax.axvline(20, color='#d62728', linestyle='--', linewidth=1, alpha=0.6,
           label='MIN_DETECTIONS = 20')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '56_detection_vs_stability.png'),
            dpi=FIG_DPI, bbox_inches='tight')
plt.show()

# Print worst offenders
print('Top 15 ecoregions by CV of peak DOY (most unstable):')
print(stab.nlargest(15, 'cv_peak_doy')[['eco_id', 'biome_name', 'mean_det', 'cv_peak_doy']]
      .to_string(index=False))


In [ ]:
# 5.7  PEAK OUTSIDE WINDOW DIAGNOSTIC ----------------------------------------------------
# How many ecoregion-years have peak DOY falling outside the onset–end window?
# Cluster by biome and detection level to see if it's a sparse-data artifact.

pow_df = master_df[['eco_id', 'eco_name', 'biome_name', 'year',
                     'peak_outside_window', 'n_detections']].copy()

n_pow   = pow_df['peak_outside_window'].sum()
pct_pow = n_pow / len(pow_df) * 100

print(f'Peak outside window: {n_pow} / {len(pow_df)} ecoregion-years ({pct_pow:.1f}%)')
print()

# By biome
pow_biome = (
    pow_df.groupby('biome_name')
    .agg(
        n_total = ('eco_id', 'count'),
        n_pow   = ('peak_outside_window', 'sum'),
    )
    .reset_index()
)
pow_biome['pct'] = round(pow_biome['n_pow'] / pow_biome['n_total'] * 100, 1)
pow_biome = pow_biome.sort_values('pct', ascending=False)

print('Peak outside window rate by biome:')
print(pow_biome.to_string(index=False))
print()

# Detection level comparison
pow_cases    = pow_df[pow_df['peak_outside_window'] == 1]['n_detections']
normal_cases = pow_df[pow_df['peak_outside_window'] == 0]['n_detections']

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(normal_cases, bins=50, alpha=0.6, label='Peak inside window', color='#1f77b4', density=True)
if len(pow_cases) > 0:
    ax.hist(pow_cases, bins=50, alpha=0.6, label='Peak outside window', color='#d62728', density=True)
ax.set_xlabel('Annual fire detections', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('5.7  Detection Counts: Peak Inside vs Outside Window',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xscale('log')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '57_peak_outside_window.png'),
            dpi=FIG_DPI, bbox_inches='tight')
plt.show()


In [ ]:
# 5.8  DIP TEST VS BC AGREEMENT -----------------------------------------------------------
# Scatter of bimodality coefficient (BC) vs dip test p-value.
# Shows whether the two methods agree or capture different phenomena.
# Quadrants:
#   Top-left:     BC > threshold, dip p > threshold → BC-only flagged
#   Bottom-right: BC < threshold, dip p < threshold → Dip-only flagged
#   Bottom-left:  Both flagged
#   Top-right:    Neither flagged

bim = master_df[['eco_id', 'biome_name', 'year', 'bc', 'dip_pval',
                  'bimodal_flag_bc', 'bimodal_flag_dip', 'n_detections']].copy()
bim = bim.dropna(subset=['bc', 'dip_pval'])

print(f'Ecoregion-years with both BC and dip: {len(bim)}')

fig, ax = plt.subplots(figsize=(9, 8))

# Color by agreement
both   = (bim['bimodal_flag_bc'] == 1) & (bim['bimodal_flag_dip'] == 1)
bc_only = (bim['bimodal_flag_bc'] == 1) & (bim['bimodal_flag_dip'] == 0)
dip_only = (bim['bimodal_flag_bc'] == 0) & (bim['bimodal_flag_dip'] == 1)
neither  = (bim['bimodal_flag_bc'] == 0) & (bim['bimodal_flag_dip'] == 0)

for mask, label, color in [
    (neither,  'Neither',  '#1f77b4'),
    (bc_only,  'BC only',  '#ff7f0e'),
    (dip_only, 'Dip only', '#2ca02c'),
    (both,     'Both',     '#d62728'),
]:
    ax.scatter(
        bim.loc[mask, 'bc'], bim.loc[mask, 'dip_pval'],
        s=8, alpha=0.4, c=color, label=f'{label} (n={mask.sum()})',
        edgecolors='none',
    )

# Threshold lines
ax.axvline(BC_THRESHOLD, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axhline(0.05, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(BC_THRESHOLD + 0.01, 1.02, f'BC = {BC_THRESHOLD}', fontsize=8,
        color='grey', transform=ax.get_xaxis_transform())
ax.text(1.02, 0.05, 'dip p = 0.05', fontsize=8, color='grey',
        transform=ax.get_yaxis_transform(), va='center')

ax.set_xlabel('Bimodality Coefficient (BC)', fontsize=11)
ax.set_ylabel('Dip Test p-value', fontsize=11)
ax.set_title('5.8  BC vs Dip Test Agreement',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, markerscale=2)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '58_bc_vs_dip_agreement.png'),
            dpi=FIG_DPI, bbox_inches='tight')
plt.show()

# Cross-tabulation
ct = pd.crosstab(
    bim['bimodal_flag_bc'].map({0: 'BC clean', 1: 'BC flagged'}),
    bim['bimodal_flag_dip'].map({0: 'Dip clean', 1: 'Dip flagged'}),
)
print()
print('Cross-tabulation (ecoregion-years):')
print(ct.to_string())


## Section 4: Summary Report

In [ ]:
# SUMMARY REPORT ----------------------------------------------------------------------------------

try:
    _ = fail_index
except NameError:
    fail_index = set()
    print('Warning: Section 1 (sanity checks) was not run.')
    print('         any_sanity_fail will be False for all ecoregions.')
    print()

fail_eco_ids = {eco_id for (eco_id, _yr) in fail_index}

summary_rows = []

for _, q_row in quality_df.iterrows():
    eco_id = q_row['eco_id']

    ref        = eco_ref[eco_ref['eco_id'] == eco_id]
    eco_name   = ref['eco_name'].iloc[0]   if len(ref) else ''
    biome_name = ref['biome_name'].iloc[0] if len(ref) else ''

    rows_eco        = master_df[master_df['eco_id'] == eco_id]
    n_years_valid   = int(rows_eco['n_years_valid'].iloc[0])    if len(rows_eco) else 0
    pct_years_valid = float(rows_eco['pct_years_valid'].iloc[0]) if len(rows_eco) else 0.0

    flag_eco        = int(q_row['bimodal_flag_eco'])
    frac_flagged    = float(q_row['frac_flagged'])
    mean_corr       = float(q_row['mean_profile_corr']) \
                      if pd.notna(q_row['mean_profile_corr']) else None
    cv_peak         = float(q_row['cv_peak_doy']) \
                      if pd.notna(q_row['cv_peak_doy'])       else None
    any_sanity_fail = eco_id in fail_eco_ids

    if any_sanity_fail:
        status = 'CHECK'
    elif flag_eco == 1:
        status = 'FLAGGED'
    else:
        status = 'CLEAN'

    summary_rows.append({
        'eco_id'            : eco_id,
        'eco_name'          : eco_name,
        'biome_name'        : biome_name,
        'n_years_valid'     : n_years_valid,
        'pct_years_valid'   : pct_years_valid,
        'mean_profile_corr' : mean_corr,
        'cv_peak_doy'       : cv_peak,
        'bimodal_flag_eco'  : flag_eco,
        'frac_flagged'      : frac_flagged,
        'any_sanity_fail'   : int(any_sanity_fail),
        'recommended_status': status,
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(output_dir, '_validation_summary.csv')
summary_df.to_csv(summary_path, index=False)

print(summary_df[[
    'eco_id', 'eco_name', 'n_years_valid',
    'bimodal_flag_eco', 'recommended_status'
]].to_string(index=False))

print('=' * 68)
print('VALIDATION SUMMARY')
print('=' * 68)
status_counts = summary_df['recommended_status'].value_counts()
for status, count in status_counts.sort_index().items():
    pct = count / len(summary_df) * 100
    print(f'  {status:<12} : {count:>4}  ({pct:.1f}%)')
print('-' * 68)
print(f'  Total        : {len(summary_df):>4}  ecoregions')
print('=' * 68)
print()
print(f'Saved: {os.path.abspath(summary_path)}')
print()